Problem Statement
Build a Simple RNN for Text Classification — Keep just the text and label columns. Tokenize with Keras Tokenizer using the top 5,000 words, then pad/truncate every sequence to exactly 50 tokens (pad_sequences(..., maxlen=50)) — this one setting works for every row regardless of the original text length. Build this exact model: Embedding(5000, 32) → SimpleRNN(32) → Dense(1, activation="sigmoid") (use Dense(n_classes, activation="softmax") if your label has more than 2 classes). Train for 5 epochs and report test accuracy. In 1–2 sentences, explain why a plain RNN can struggle once a sequence gets long (the vanishing gradient problem).
https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection

Import libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Load the JSON dataset

The dataset is JSON-lines formatted, so use lines=True.

In [2]:
df = pd.read_json(
    "Sarcasm_Headlines_Dataset.json",
    lines=True
)

df.head()

,article_link,headline,is_sarcastic
0,https://www.huffingtonpost.com/entry/versace-b...,former versace store clerk sues over secret 'b...,0
1,https://www.huffingtonpost.com/entry/roseanne-...,the 'roseanne' revival catches up to our thorn...,0
2,https://local.theonion.com/mom-starting-to-fea...,mom starting to fear son's web series closest ...,1
3,https://politics.theonion.com/boehner-just-wan...,"boehner just wants wife to listen, not come up...",1
4,https://www.huffingtonpost.com/entry/jk-rowlin...,j.k. rowling wishes snape happy birthday in th...,0


Check the columns:

In [3]:
print(df.columns)
print(df.shape)

Index(['article_link', 'headline', 'is_sarcastic'], dtype='object')
(26709, 3)


Keep only text and label

In [4]:
df = df[["headline", "is_sarcastic"]]

df.head()

,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0


Check for missing values:

In [5]:
print(df.isnull().sum())

headline        0
is_sarcastic    0
dtype: int64


Separate X and y

In [6]:
X = df["headline"]
y = df["is_sarcastic"]

In [7]:
print(X.head())
print(y.head())

0    former versace store clerk sues over secret 'b...
1    the 'roseanne' revival catches up to our thorn...
2    mom starting to fear son's web series closest ...
3    boehner just wants wife to listen, not come up...
4    j.k. rowling wishes snape happy birthday in th...
Name: headline, dtype: object
0    0
1    0
2    1
3    1
4    0
Name: is_sarcastic, dtype: int64


Train/test split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Tokenization

In [9]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X_train)

In [10]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

Pad every sequence to exactly 50 tokens

In [12]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=50
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=50
)

Check the shape:

In [13]:
print(X_train_pad.shape)
print(X_test_pad.shape)

(21367, 50)
(5342, 50)


Build the exact RNN

In [14]:
model = Sequential([

    Embedding(5000, 32),

    SimpleRNN(32),

    Dense(1, activation="sigmoid")
])

Check the model:

In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Compile the model

In [16]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Train for exactly 5 epochs

In [17]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    validation_split=0.2,
    batch_size=32
)

Epoch 1/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.7759 - loss: 0.4537 - val_accuracy: 0.8498 - val_loss: 0.3442
Epoch 2/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.8962 - loss: 0.2545 - val_accuracy: 0.8386 - val_loss: 0.3603
Epoch 3/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.9424 - loss: 0.1541 - val_accuracy: 0.8435 - val_loss: 0.4150
Epoch 4/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9748 - loss: 0.0796 - val_accuracy: 0.8304 - val_loss: 0.5140
Epoch 5/5
535/535 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9864 - loss: 0.0439 - val_accuracy: 0.8264 - val_loss: 0.6147


Test accuracy

Now evaluate on the unseen test data.

In [18]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8255 - loss: 0.5925
Test Accuracy: 82.55%


See predictions

In [19]:
predictions = model.predict(X_test_pad)

predicted_labels = (predictions > 0.5).astype(int).flatten()

167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


vanishing-gradient

A plain RNN can struggle with long sequences because information from earlier words can gradually fade as it passes through many time steps. This is called the vanishing gradient problem, and it makes it difficult for the RNN to learn long-term dependencies.